# Step 7 — Conformational State Comparison (Corrected)

**Corrected version** with Drive I/O fix (copies JSON files to local /tmp before reading).

Synthesises Steps 4, 5, and 6 into the paper's central finding:
does the structural mimicry signal depend on which spike conformational state is examined?

In [ ]:
import json, os, shutil
import numpy as np

MOUNT_DRIVE = True
BASE_DIR = "/content"
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/myocarditis_pipeline"

STEP4_DIR = os.path.join(BASE_DIR, "step4")
STEP5_DIR = os.path.join(BASE_DIR, "step5")
STEP6_DIR = os.path.join(BASE_DIR, "step6")
STEP7_DIR = os.path.join(BASE_DIR, "step7")
os.makedirs(STEP7_DIR, exist_ok=True)

# Copy to local /tmp to avoid Drive I/O errors on large reads
LOCAL_TMP = "/tmp/step7_inputs"
os.makedirs(LOCAL_TMP, exist_ok=True)

files_to_load = {
    "step4": os.path.join(STEP4_DIR, "alignment_results.json"),
    "step5": os.path.join(STEP5_DIR, "validation_results.json"),
    "step6": os.path.join(STEP6_DIR, "antigenicity_results.json"),
}

loaded = {}
for key, src_path in files_to_load.items():
    local_path = os.path.join(LOCAL_TMP, f"{key}.json")
    print(f"Copying {key} to local storage...", end=" ")
    shutil.copy2(src_path, local_path)
    with open(local_path) as f:
        loaded[key] = json.load(f)
    print("OK")

step4 = loaded["step4"]
step5 = loaded["step5"]
step6 = loaded["step6"]
print("\nAll prior step outputs loaded.\n")

# LAYER 1
print("=" * 80)
print("LAYER 1: Whole-domain structural alignment (Step 4)")
print("=" * 80)
wd_results = step4["results"]
antigens = sorted(set(r["antigen"] for r in wd_results))
states = ["closed", "one_rbd_up", "postfusion"]

print(f"\n{'Antigen':<8}", end="")
for s in states:
    print(f"  {s:>15}", end="")
print(f"  {'delta':>8}  {'state-dependent?':>18}")
print("-" * 85)

wd_summary = {}
for antigen in antigens:
    scores = {}
    for s in states:
        hit = [r for r in wd_results if r["antigen"] == antigen and r["spike_state"] == s]
        scores[s] = hit[0]["tm_score"] if hit and hit[0]["tm_score"] is not None else 0.0
    vals = list(scores.values())
    delta = max(vals) - min(vals)
    best_state = max(scores, key=scores.get)
    dependent = "YES" if delta > 0.05 else "no"
    wd_summary[antigen] = {"scores": scores, "delta": delta, "best_state": best_state}
    print(f"{antigen:<8}", end="")
    for s in states:
        marker = " *" if s == best_state and delta > 0.05 else "  "
        print(f"  {scores[s]:>13.4f}{marker}", end="")
    print(f"  {delta:>8.4f}  {dependent:>18}")

# LAYER 2
print("\n" + "=" * 80)
print("LAYER 2: Statistical validation (Step 5)")
print("=" * 80)
print(f"\n{'Antigen':<8} {'State':<15} {'Obs TM':>8} {'p-value':>10} {'Sig?':>6}")
print("-" * 50)
for p in step5["pvalue_results"]:
    sig = "YES" if p["empirical_pvalue"] < 0.05 else "no"
    print(f"{p['antigen']:<8} {p['spike_state']:<15} {p['observed_tm']:>8.4f} "
          f"{p['empirical_pvalue']:>10.4f} {sig:>6}")

# LAYER 3
print("\n" + "=" * 80)
print("LAYER 3: Fragment-level patch analysis")
print("=" * 80)
frag = step4.get("fragment_analysis", {})
patches = frag.get("significant_patches", [])
valid_patches = [p for p in patches if p["tm_score"] <= 1.0 and p["aln_length"] >= 15]
print(f"\nValid fragment hits: {len(valid_patches)}")
for antigen in antigens:
    for state in states:
        sp = [p for p in valid_patches if p.get("antigen") == antigen and p.get("spike_state") == state]
        if sp:
            tms = [p["tm_score"] for p in sp]
            print(f"  {antigen} {state}: {len(sp)} patches, TM [{min(tms):.3f}-{max(tms):.3f}]")

# LAYER 4
print("\n" + "=" * 80)
print("LAYER 4: Surface accessibility (Step 6)")
print("=" * 80)
patch_details = step6.get("patch_exposure_details", [])
n_total = step6["n_patches_analyzed"]
n_exposed = step6["n_both_sides_exposed"]
print(f"\nPatches analyzed: {n_total}, both sides exposed: {n_exposed}")
for antigen in ["TPM1", "MYH6"]:
    ag = [p for p in patch_details if p.get("antigen") == antigen]
    ag_exp = [p for p in ag if p.get("both_sides_exposed")]
    if ag:
        print(f"  {antigen}: {len(ag_exp)}/{len(ag)} bilaterally exposed")

# INTEGRATED FINDING
print("\n" + "=" * 80)
print("INTEGRATED FINDING")
print("=" * 80)
print("""
Postfusion spike S2 coiled-coil (HR1/HR2) is structurally near-identical to cardiac
tropomyosin (TPM1) at the epitope-relevant 30-residue scale (TM up to 0.998, LDDT 0.99).

  1. CONFORMATION-DEPENDENT (postfusion TM=0.53 > prefusion TM=0.41-0.43)
  2. NOT CARDIAC-SPECIFIC (p=0.072, driven by shared coiled-coil class)
  3. ANTIBODY-ACCESSIBLE on both sides for TPM1 (93% bilaterally exposed)
  4. NOT accessible for MYH6 (83% buried on the MYH6 side)

Mechanism: structural-class cross-reactivity, not specific molecular mimicry.
""")

# Write output
from datetime import datetime, timezone
comparison_output = {
    "pipeline_step": 7,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "generated_in": "Google Colab",
    "whole_domain_summary": wd_summary,
    "statistical_validation": {"any_significant": False,
        "closest": {"antigen": "TPM1", "state": "postfusion", "tm": 0.5267, "pvalue": 0.072}},
    "fragment_summary": {"n_valid_patches": len(valid_patches)},
    "surface_accessibility": {"n_analyzed": n_total, "n_both_exposed": n_exposed,
        "tpm1_rate": f"{len([p for p in patch_details if p.get('antigen')=='TPM1' and p.get('both_sides_exposed')])} / {len([p for p in patch_details if p.get('antigen')=='TPM1'])}",
        "myh6_rate": f"{len([p for p in patch_details if p.get('antigen')=='MYH6' and p.get('both_sides_exposed')])} / {len([p for p in patch_details if p.get('antigen')=='MYH6'])}"},
    "integrated_finding": "Postfusion S2 coiled-coil structurally mimics TPM1 surfaces "
        "(conformation-dependent, bilaterally exposed, not cardiac-specific). "
        "Mechanism: structural-class cross-reactivity, not specific molecular mimicry.",
}
output_path = os.path.join(STEP7_DIR, "conformational_comparison.json")
with open(output_path, "w") as f:
    json.dump(comparison_output, f, indent=2, default=str)
print(f"Written to: {output_path}")